In [ ]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModel,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

In [ ]:
dataset = load_dataset("sentence-transformers/all-nli", "pair-class")
print(dataset)

In [ ]:
model_id = "model"
embed_model_id = "embed_model"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModel.from_pretrained(model_id)

In [ ]:
def keep_good(example):
    return example["label"] == 0   # entailment فقط = positive pairs

dataset = dataset.filter(keep_good)

def keep_cols(example):
    return {
        "sentence1": example["premise"],
        "sentence2": example["hypothesis"],
    }

dataset = dataset.map(keep_cols)
dataset = dataset.remove_columns(
    [c for c in dataset["train"].column_names if c not in ["sentence1", "sentence2"]]
)

print(dataset)

In [ ]:
max_length = 128

def tokenize(examples):
    a = tokenizer(
        examples["sentence1"],
        truncation=True,
        max_length=max_length,
        padding=False,
    )
    b = tokenizer(
        examples["sentence2"],
        truncation=True,
        max_length=max_length,
        padding=False,
    )

    return {
        "input_ids_a": a["input_ids"],
        "attention_mask_a": a["attention_mask"],
        "input_ids_b": b["input_ids"],
        "attention_mask_b": b["attention_mask"],
    }

dataset = dataset.map(
    tokenize,
    batched=True,
    remove_columns=["sentence1", "sentence2"],
)

dataset.set_format("torch")
print(dataset)

In [ ]:
class PairCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        a_features = [
            {
                "input_ids": f["input_ids_a"],
                "attention_mask": f["attention_mask_a"],
            }
            for f in features
        ]
        b_features = [
            {
                "input_ids": f["input_ids_b"],
                "attention_mask": f["attention_mask_b"],
            }
            for f in features
        ]

        batch_a = self.tokenizer.pad(a_features, return_tensors="pt")
        batch_b = self.tokenizer.pad(b_features, return_tensors="pt")

        batch_size = batch_a["input_ids"].size(0)

        return {
            "input_ids_a": batch_a["input_ids"],
            "attention_mask_a": batch_a["attention_mask"],
            "input_ids_b": batch_b["input_ids"],
            "attention_mask_b": batch_b["attention_mask"],
            "labels": torch.zeros(batch_size, dtype=torch.long),  # dummy
        }

In [ ]:
def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts

In [ ]:
class ContrastiveTrainer(Trainer):
    def __init__(self, *args, temperature=0.05, **kwargs):
        super().__init__(*args, **kwargs)
        self.temperature = temperature

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        inputs = inputs.copy()
        inputs.pop("labels", None)

        out_a = model(
            input_ids=inputs["input_ids_a"],
            attention_mask=inputs["attention_mask_a"],
        )
        out_b = model(
            input_ids=inputs["input_ids_b"],
            attention_mask=inputs["attention_mask_b"],
        )

        emb_a = mean_pooling(out_a.last_hidden_state, inputs["attention_mask_a"])
        emb_b = mean_pooling(out_b.last_hidden_state, inputs["attention_mask_b"])

        emb_a = F.normalize(emb_a, p=2, dim=1)
        emb_b = F.normalize(emb_b, p=2, dim=1)

        logits = torch.matmul(emb_a, emb_b.T) / self.temperature
        targets = torch.arange(logits.size(0), device=logits.device)

        loss_a = F.cross_entropy(logits, targets)
        loss_b = F.cross_entropy(logits.T, targets)
        loss = (loss_a + loss_b) / 2

        return (loss, {"logits": logits}) if return_outputs else loss

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        inputs = self._prepare_inputs(inputs)

        with torch.no_grad():
            loss, outputs = self.compute_loss(model, inputs, return_outputs=True)

        loss = loss.detach()
        logits = outputs["logits"].detach()

        if prediction_loss_only:
            return (loss, None, None)

        labels = torch.arange(logits.size(0), device=logits.device)
        return (loss, logits, labels)

In [ ]:
training_args = TrainingArguments(
    output_dir=embed_model_id,
    learning_rate=2e-5,
    weight_decay=0.01,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    eval_strategy="steps",
    save_strategy="steps",
    logging_strategy="steps",
    logging_steps=200,
    eval_steps=500,
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    fp16=True,
    seed=42,
    remove_unused_columns=False,
)

trainer = ContrastiveTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["dev"],
    data_collator=PairCollator(tokenizer),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

In [ ]:
trainer.train()

In [ ]:
trainer.save_model(embed_model_id)

In [ ]:
print("Best checkpoint:", trainer.state.best_model_checkpoint)


In [ ]:
logs = trainer.state.log_history

train_steps = [log["step"] for log in logs if "loss" in log and "eval_loss" not in log]
train_losses = [log["loss"] for log in logs if "loss" in log and "eval_loss" not in log]

eval_steps = [log["step"] for log in logs if "eval_loss" in log]
eval_losses = [log["eval_loss"] for log in logs if "eval_loss" in log]

plt.figure(figsize=(12, 6))
plt.plot(train_steps, train_losses, label="Training Loss", linewidth=2)
plt.plot(eval_steps, eval_losses, label="Validation Loss", linewidth=2)
plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.title("Contrastive Training Loss")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.3)
plt.show()

In [ ]:
metrics = trainer.evaluate(dataset["test"])
print(metrics)

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F

embed_model_id = "embed_model"

tokenizer = AutoTokenizer.from_pretrained(embed_model_id)
model = AutoModel.from_pretrained(embed_model_id).eval().cuda()


def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts


def encode(texts):
    batch = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        out = model(**batch)
        emb = mean_pooling(out.last_hidden_state, batch["attention_mask"])
        emb = F.normalize(emb, p=2, dim=1)

    return emb


pairs = [
    ("A dog is an animal.", "A cat is an animal."),
    ("Paris is the capital of France.", "France has Paris as its capital."),
    ("I love machine learning.", "The sky is blue."),
    ("Microsoft is a company.", "Google is a company."),
    ("He is eating an apple.", "She is driving a car."),
]

for a, b in pairs:
    emb = encode([a, b])
    sim = F.cosine_similarity(emb[0].unsqueeze(0), emb[1].unsqueeze(0)).item()

    print(f"Text 1: {a}")
    print(f"Text 2: {b}")
    print(f"Similarity: {sim:.4f}")
    print("-" * 60)